# Rytmi — rhythm learning for dance and music with Gemma 4

A submission for the **Gemma 4 Good Hackathon** (*Future of Education* track).

Rytmi helps dancers and musicians **hear the beat, find the "1", and feel
phrase structure** in music — especially in styles like kizomba where the
pulse is subtle and easy to lose.  It pairs **librosa DSP** (beat tracking,
onset detection, section/phase segmentation) with **Gemma 4** (per-section
coaching, transitions, listening guides, drills).

| Link | |
|---|---|
| Code (public mirror) | https://github.com/mukatee/rytmi-pub |
| Kaggle Notebook      | https://www.kaggle.com/code/donkeys/gemma-dancing |
| Demo video           | https://youtu.be/ISkf6fZbG-Y |
| Writeup              | see `docs/kaggle_writeup.md` in the repo |

**Audio licence:** the demo track used below is released under
[**CC0 1.0 Universal**](https://creativecommons.org/publicdomain/zero/1.0/)
(public domain) from the album *"Un Toque De To"* on the Internet
Archive.  See the attribution cell at the bottom for full credit.


## How to read this notebook

> **You don't need to run anything to evaluate.**  Cached Gemma outputs
> are displayed inline below every prompt.  Hit *Run All* only if you
> want to re-render the interactive timeline widgets (which need the
> audio file present) or generate fresh Gemma text in live mode.

**Architecture in one line:**

```
audio  ─►  librosa DSP  ─►  RhythmAnalysis  ─►  Gemma 4 prompt  ─►  coaching text
            (beats,         (tempo, beats,        (style-aware,
             onsets,         downbeats,            grounded in
             sections)       phases, clarity)      DSP features)
```

**Why this split:** DSP gives Gemma reliable, *measurable* rhythm
features.  Gemma turns those into language a learner can act on.
Verifiers check that Gemma's output covers the song without inventing
sections.  We deliberately do **not** ask Gemma to detect the beat from
raw audio — DSP is better at that.

**5-minute skim path:**
*cells 1 → 2 → 9 → first track section → summary*.

**Modes:** cached (default, no setup) · live (Gemma API key) · bring
your own audio.  See cell 7 for configuration.


In [1]:
# Runtime dependencies.  Quiet mode keeps Kaggle output tidy.
# `standard-aifc` / `standard-sunau` are needed because `audioop` was
# removed in Python 3.13 and librosa still imports them indirectly.
!pip install -q \
    "librosa>=0.11" "scipy>=1.14" "numpy>=2.0" \
    "matplotlib>=3.8" "soundfile" "openai" \
    "standard-aifc; python_version>='3.13'" \
    "standard-sunau; python_version>='3.13'" \
    requests

In [2]:
# Install Rytmi straight from the public GitHub mirror — no need to
# attach the repo as a Kaggle Dataset.
!pip install -q "git+https://github.com/mukatee/rytmi-pub.git" 

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
# Pull the demo CC0 1.0 Universal track from the Internet Archive.
# The album is fetched as a single zip (more reliable than per-file URLs).
# Skips the download if the target is already present.
import io, os, sys, zipfile
from pathlib import Path
import requests

# Kaggle drops user-writable space at /kaggle/working; locally we use
# the repo's data/samples/kizomba_cc dir if it exists.
IS_KAGGLE = Path("/kaggle/working").exists()
if IS_KAGGLE:
    AUDIO_DIR = Path("/kaggle/working/cc0_audio")
else:
    repo_audio = Path("data/samples/kizomba_cc").resolve()
    AUDIO_DIR = repo_audio if repo_audio.exists() else Path("cc0_audio").resolve()
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
print(f"audio dir: {AUDIO_DIR}")

# Target filenames expected by the rest of the notebook, keyed by track-number prefix.
PREFIX_TO_TARGET = {
    "07": "07_besame_otra_vez.mp3",
}

targets = {t: AUDIO_DIR / t for t in PREFIX_TO_TARGET.values()}

# Check which files are already present.
missing_prefixes = [
    prefix for prefix, tname in PREFIX_TO_TARGET.items()
    if not (AUDIO_DIR / tname).exists() or (AUDIO_DIR / tname).stat().st_size < 100_000
]

for tname, dest in targets.items():
    if dest.exists() and dest.stat().st_size > 100_000:
        print(f"  ok  {tname}  ({dest.stat().st_size:,} bytes, already present)")

if missing_prefixes:
    ZIP_URL = (
        "https://archive.org/compress/UnToqueDTo"
        "/formats=VBR%20MP3&file=/UnToqueDTo.zip"
    )
    print(f"  downloading album zip from archive.org …", flush=True)
    with requests.get(ZIP_URL, stream=True, timeout=300) as r:
        r.raise_for_status()
        data = r.content
    print(f"  zip downloaded ({len(data):,} bytes), extracting …")

    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        mp3_members = [n for n in zf.namelist() if n.lower().endswith(".mp3")]
        for prefix in missing_prefixes:
            # Find the zip entry whose filename starts with the two-digit prefix.
            match = next(
                (n for n in mp3_members if Path(n).name.startswith(prefix)),
                None,
            )
            if match is None:
                print(f"  WARNING: no file with prefix {prefix!r} found in zip")
                continue
            dest = AUDIO_DIR / PREFIX_TO_TARGET[prefix]
            dest.write_bytes(zf.read(match))
            print(f"  wrote {dest.name}  ({dest.stat().st_size:,} bytes)  ← {Path(match).name}")

print("audio ready.")

audio dir: /kaggle/working/cc0_audio
  downloading album zip from archive.org …
  zip downloaded (118,986,961 bytes), extracting …
  wrote 07_besame_otra_vez.mp3  (15,709,266 bytes)  ← 07-BesameOtraVezconKla-cUnToqueDTo------waveMusic------.mp3
audio ready.


### Bring your own audio (optional)

To run the pipeline on your own track instead of (or in addition to)
the CC0 tracks above:

1. Right sidebar → **Add Input → Datasets** → attach an existing Kaggle
   dataset that contains an MP3 or WAV, or upload your own.
2. In **cell 7 (config)** set `AUDIO_PATH = "/kaggle/input/<dataset>/<file>.mp3"`.
3. Make sure `MODE = "live"` (needs a `GEMMA_API_KEY` Kaggle Secret).
4. Run cells 1–7, then jump to the *Bring your own audio* cell near
   the bottom.

Rytmi will analyze the new track and stream a full Gemma coaching pass
just like the cached tracks below.


In [4]:
# === Configuration ============================================================
# Three modes:
#   "cached" → read every Gemma output from notebooks/cache/  (no API key)
#   "live"   → call Gemma fresh for every prompt              (needs API key)
#   "auto"   → live if a key is present, cached otherwise
MODE = "auto"

# Gemma 4 model.  Default is the public HuggingFace id for the E4B
# instruction-tuned checkpoint -- well known, easy to look up, and small
# enough to self-host on Kaggle's free T4.  Override with whatever slug
# your endpoint expects (e.g. Ollama: 'gemma4:e4b';  Google AI Studio:
# 'models/gemma-4-e4b-it';  larger HF checkpoints: 'google/gemma-4-26b-a4b-it').
GEMMA_MODEL    = "google/gemma-4-e4b-it"
GEMMA_API_BASE = "https://generativelanguage.googleapis.com/v1beta/openai/"

# Bring-your-own audio — set to /kaggle/input/<dataset>/<file>.mp3 to run
# the live pipeline on your own track at the bottom of the notebook.
AUDIO_PATH = None

# === API key resolution =======================================================
# Prefers Kaggle Secret → env var.  Never falls back to disk.
import os
GEMMA_API_KEY = None
try:
    from kaggle_secrets import UserSecretsClient
    GEMMA_API_KEY = UserSecretsClient().get_secret("GEMMA_API_KEY")
    print("found GEMMA_API_KEY via Kaggle Secret")
except Exception:
    GEMMA_API_KEY = os.environ.get("GEMMA_API_KEY")
    if GEMMA_API_KEY:
        print("found GEMMA_API_KEY via environment")

if MODE == "auto":
    MODE = "live" if GEMMA_API_KEY else "cached"

print(f"resolved MODE = {MODE!r}")
print(f"model        = {GEMMA_MODEL!r}")
print(f"endpoint     = {GEMMA_API_BASE!r}")

# A quick reminder for judges:
if MODE == "cached":
    print()
    print("Cached mode: all Gemma outputs below are pre-generated.")
    print("To run live, add a Kaggle Secret named GEMMA_API_KEY and re-run from here.")
    print("API key: https://aistudio.google.com/apikey")

resolved MODE = 'cached'
model        = 'google/gemma-4-e4b-it'
endpoint     = 'https://generativelanguage.googleapis.com/v1beta/openai/'

Cached mode: all Gemma outputs below are pre-generated.
To run live, add a Kaggle Secret named GEMMA_API_KEY and re-run from here.
API key: https://aistudio.google.com/apikey


In [5]:
# Setup: cache loader, optional Gemma client, Kaggle-aware viz patch.
import json
from pathlib import Path

from IPython.display import HTML, Markdown, display

from rytmi import analyze, load_audio
from rytmi import viz as rytmi_viz
from rytmi.dsp import describe_sections, describe_transitions
from rytmi.prompts import (
    QUESTION_KIZOMBA_DRILLS,
    QUESTION_KIZOMBA_TRANSITIONS,
    QUESTION_KIZOMBA_TUTOR,
    QUESTION_LISTENING_GUIDE,
    QUESTION_RHYTHM_ANATOMY,
    QUESTION_SONG_ARC,
    format_unified_timeline,
)

# ---------------------------------------------------------------------------
# Cache directory — committed under notebooks/cache/ in the public repo.
# Both Kaggle (after `pip install git+...`) and local checkouts need to
# find it.  On Kaggle we re-download via git into /kaggle/working/_cache
# only if the pip-installed wheel didn't ship it (the public mirror keeps
# the cache files in-tree).
# ---------------------------------------------------------------------------
def _find_cache_dir() -> Path:
    candidates = [
        Path.cwd() / "notebooks" / "cache",        # local: notebook at repo root
        Path.cwd().parent / "notebooks" / "cache", # local: notebook in notebooks/
        Path("/kaggle/working/cache"),
    ]
    for c in candidates:
        if (c / "cc0_09_te_quiero_amor_outputs.json").exists():
            return c
    # Fallback: clone the public repo into /kaggle/working/_rytmi_cache
    target = Path("/kaggle/working/_rytmi_cache") if IS_KAGGLE else Path.cwd() / "_rytmi_cache"
    if not (target / "notebooks" / "cache" / "cc0_09_te_quiero_amor_outputs.json").exists():
        print(f"  cache not found in repo, cloning rytmi-pub into {target} …")
        import subprocess
        subprocess.run(
            ["git", "clone", "--depth=1",
             "https://github.com/mukatee/rytmi-pub.git", str(target)],
            check=True,
        )
    return target / "notebooks" / "cache"

CACHE_DIR = _find_cache_dir()
print(f"cache dir: {CACHE_DIR}")

# ---------------------------------------------------------------------------
# Cache file IO
# ---------------------------------------------------------------------------
def load_cached_outputs(slug: str) -> dict:
    p = CACHE_DIR / f"cc0_{slug}_outputs.json"
    return json.loads(p.read_text())

# ---------------------------------------------------------------------------
# Optional live Gemma client.  Only constructed in live mode.
# ---------------------------------------------------------------------------
GEMMA_CLIENT = None
if MODE == "live":
    from rytmi.llm import load_cloud_model, explain_rhythm  # noqa: F401
    GEMMA_CLIENT, _ = load_cloud_model(
        model_id=GEMMA_MODEL,
        base_url=GEMMA_API_BASE,
        api_key=GEMMA_API_KEY,
        print_url=True,
    )
    print(f"Gemma client ready: {GEMMA_MODEL}")

# ---------------------------------------------------------------------------
# Prompt menu — name → (question, max_tokens).  Token budgets are
# generous because some Gemma 4 checkpoints reserve a chunk of the
# completion window for internal reasoning before producing visible
# text.  Smaller models ignore the unused headroom.
# ---------------------------------------------------------------------------
PROMPTS = [
    ("rhythm_anatomy",       QUESTION_RHYTHM_ANATOMY,       4096),
    ("listening_guide",      QUESTION_LISTENING_GUIDE,      4096),
    ("song_arc",             QUESTION_SONG_ARC,             4096),
    ("kizomba_tutor",        QUESTION_KIZOMBA_TUTOR,        6144),
    ("kizomba_drills",       QUESTION_KIZOMBA_DRILLS,       6144),
    ("kizomba_transitions",  QUESTION_KIZOMBA_TRANSITIONS,  6144),
]

def run_live_outputs(analysis) -> dict:
    """Run every Gemma prompt fresh.  Used in live mode and for BYO audio."""
    from rytmi.llm import explain_rhythm
    out = {
        "_meta": {
            "duration_s": float(analysis.audio.duration),
            "tempo": float(analysis.tempo),
            "n_beats": int(len(analysis.beats.times)),
            "downbeat_confidence": float(getattr(analysis, "downbeat_confidence", 0.0) or 0.0),
            "n_sections": int(len(getattr(analysis, "sections", []) or [])),
            "n_phases": int(len(getattr(analysis, "phases", []) or [])),
            "sections_describe": describe_sections(analysis),
        },
        "_transitions_describe": describe_transitions(analysis),
    }
    for name, question, max_tok in PROMPTS:
        print(f"  Gemma → {name} …", flush=True)
        text = ""
        for budget in (max_tok, max_tok * 2):
            text = explain_rhythm(
                analysis, question=question,
                processor=GEMMA_CLIENT, model=GEMMA_MODEL,
                max_new_tokens=budget, temperature=0.5,
            )
            if text.strip():
                break
        out[name] = text
    if out.get("kizomba_tutor") and out.get("kizomba_transitions"):
        out["unified_timeline"] = format_unified_timeline(
            out["kizomba_tutor"], out["kizomba_transitions"],
        )
    return out

def get_outputs(slug: str, analysis=None) -> dict:
    """Cached-mode reader; in live mode, re-runs Gemma."""
    if MODE == "cached":
        return load_cached_outputs(slug)
    if analysis is None:
        raise ValueError("live mode needs an analysis to call Gemma against")
    return run_live_outputs(analysis)

# ---------------------------------------------------------------------------
# interactive_timeline asset patch (always on).
#
# `interactive_timeline` writes a PNG + WAV next to the installed rytmi
# package (.../site-packages/rytmi/../../data/_timeline_cache/) and
# emits HTML that references them via the relative URL
# `../data/_timeline_cache/<file>`.  That URL is resolved by the
# browser against the notebook's location, NOT against the package's
# install path — so when rytmi is installed via `pip install git+...`
# (as it is here), the widget writes to one place and the browser
# fetches from another, leaving the image blank and the audio dead.
#
# Fix: always write the assets into `<notebook_cwd>/_timeline_cache/`
# and rewrite the URL prefix to `_timeline_cache/` so the browser
# fetches them from the same place.  Works identically on Kaggle
# (/kaggle/working) and locally (notebooks/).
# ---------------------------------------------------------------------------
_PATCHED_DIR = Path.cwd() / "_timeline_cache"
_PATCHED_DIR.mkdir(parents=True, exist_ok=True)
rytmi_viz._timeline_cache_dir = lambda: _PATCHED_DIR

def show_timeline(analysis, title: str) -> None:
    html = rytmi_viz.interactive_timeline(analysis, title=title, pixels_per_second=80)
    # Same rewrite locally and on Kaggle: the assets now live next to
    # the notebook, not under ../data/.
    html = html.replace("../data/_timeline_cache/", "_timeline_cache/")
    display(HTML(html))

# ---------------------------------------------------------------------------
# Text rendering helper.  Gemma outputs use single newlines between
# lines (each `P1:`, `P2:` ... on its own line).  Markdown collapses
# single newlines into spaces within a paragraph, which turns each
# prompt's output into one long unreadable line.  Convert single
# newlines to Markdown hard breaks (two trailing spaces + newline) so
# per-line structure is kept intact, while leaving blank-line
# paragraph breaks alone.
# ---------------------------------------------------------------------------
def as_markdown(text: str) -> str:
    lines = text.split("\n")
    out = []
    for i, line in enumerate(lines):
        is_last = i == len(lines) - 1
        next_blank = (not is_last) and lines[i + 1].strip() == ""
        if line.strip() == "" or is_last or next_blank:
            out.append(line)
        else:
            out.append(line + "  ")  # trailing 2 spaces = hard break
    return "\n".join(out)

print("setup done.")

  cache not found in repo, cloning rytmi-pub into /kaggle/working/_rytmi_cache …


Cloning into '/kaggle/working/_rytmi_cache'...


cache dir: /kaggle/working/_rytmi_cache/notebooks/cache
setup done.


## Why this track?

The [demo video](https://youtu.be/ISkf6fZbG-Y) uses copyrighted
kizomba/bachata for authenticity (Filomena Maricoa, Charbel).  For this
notebook we needed audio that is **fully free to redistribute** so
judges can rerun without rights issues.  After surveying:

- **Free Music Archive** — no kizomba/zouk in any CC tier
- **Jamendo, ccMixter, SoundCloud CC filter** — same story

We settled on *Besame Otra Vez* from the **CC0 1.0 Universal** album
[*"Un Toque De To"*](https://archive.org/details/un-toque-de-to) on
Internet Archive.  At 89 BPM with a clearly audible break and peak
section it sits inside real-kizomba tempo range and gives the
kizomba-tuned DSP path something genuine to chew on — and **the
pipeline works equally well on any audio you supply**.  See the
*Bring your own audio* cell near the bottom.

Production note: real-world Rytmi users would point this at their own
practice tracks.  The CC0 substitution is a redistribution-rights
accommodation for this submission, not a product limitation.


In [6]:
# Per-track display function.  Takes a slug + display title; loads
# audio, runs DSP (always live — fast), pulls Gemma outputs from the
# cache or runs live, then renders everything inline.
def display_track(slug: str, display_title: str) -> None:
    print(f"=== {slug} ({display_title}) ===")
    audio = load_audio(str(AUDIO_DIR / f"{slug}.mp3"))
    analysis = analyze(audio, dance_style="kizomba")
    outs = get_outputs(slug, analysis if MODE == "live" else None)

    # --- DSP summary -------------------------------------------------------
    meta = outs.get("_meta", {})
    display(Markdown(
        f"**DSP summary** — tempo {analysis.tempo:.1f} BPM · "
        f"{len(analysis.beats.times)} beats · "
        f"{len(getattr(analysis, 'sections', []) or [])} sections · "
        f"{len(getattr(analysis, 'phases', []) or [])} phases · "
        f"downbeat confidence {getattr(analysis, 'downbeat_confidence', 0.0):.2f}"
    ))

    # --- Interactive timeline + audio player ------------------------------
    display(Markdown("### Interactive timeline\nPlay, scrub, or click to seek. "
                     "The red cursor tracks the audio."))
    show_timeline(analysis, title=display_title)

    # --- Per-section table -------------------------------------------------
    display(Markdown("### Section table (what DSP heard)"))
    print(describe_sections(analysis))

    # --- Gemma outputs -----------------------------------------------------
    headings = {
        "rhythm_anatomy":      "Rhythm anatomy — kizomba as a genre",
        "listening_guide":     "Listening guide — orient the ear",
        "song_arc":            "Song arc — the energy journey",
        "kizomba_tutor":       "Kizomba tutor — per-phase coaching",
        "kizomba_drills":      "Practice plan — drills tied to sections",
        "kizomba_transitions": "Transitions — what to do at boundaries",
    }
    for key, heading in headings.items():
        text = outs.get(key, "")
        display(Markdown(f"### {heading}"))
        if text and not text.startswith("[ERROR"):
            display(Markdown(as_markdown(text)))
        else:
            display(Markdown(f"_(no cached output for `{key}` — re-run cache build)_"))

    if "unified_timeline" in outs:
        display(Markdown("### Unified timeline (phases + transitions in one story)"))
        display(Markdown("```\n" + outs["unified_timeline"] + "\n```"))

print("display_track() ready.")

display_track() ready.


## *Besame Otra Vez* — full pipeline

In [7]:
display_track('07_besame_otra_vez', 'Besame Otra Vez')

=== 07_besame_otra_vez (Besame Otra Vez) ===


**DSP summary** — tempo 89.1 BPM · 472 beats · 30 sections · 12 phases · downbeat confidence 0.01

### Interactive timeline
Play, scrub, or click to seek. The red cursor tracks the audio.

### Section table (what DSP heard)

Section table — 89 BPM, 4/4, 8-count phrases, 30 sections
  downbeat offset = 2 (confidence = 0.01)
 # label            energy  time (s)        P/M            drift      dur           signals (RMS×, opb, H×, P×)
------------------------------------------------------------------------------------------------------------------------
 0 intro            low       0.0– 14.5     P0/M0→P3/M5    —           14.5s/ 2.7ph RMS×0.61 opb=2.7 H×0.59 P×0.42 BC=0.42
 1 short_break[percussive] medium   14.5– 21.4     P3/M5→P4/M7    -5.7b*       6.9s/ 1.3ph RMS×0.88 opb=2.1 H×1.05 P×0.64 BC=0.26
 2 main             medium   21.4– 26.8     P4/M7→P5/M9    -2.1b*       5.4s/ 1.0ph RMS×1.02 opb=2.6 H×0.96 P×1.10 BC=0.48
 3 main             medium   26.8– 42.8     P5/M9→P8/M15   +2.2b*      16.0s/ 3.0ph RMS×1.03 opb=2.5 H×0.96 P×1.06 BC=0.48
 4 main             medium   42.8– 53.4     P8/M15→P10/M19 -2.1b*      10.7s/ 2.0ph RMS×1.08 opb=2.1 H×1.02 P×1.12 BC=0.41
 5 main             medium   53.4– 64.1     P

...

P1: intro (0s-15s, beat: clear) — Drill: maintain a close embrace and feel the pulse together. 15s.  
P2: short_break (15s-21s, beat: moderate) — Drill: keep steps small and test the pulse. 6s.  
P3-P6: main (21s-155s, beat: clear) — Drill: practice a steady walk-step, letting the bass guide your weight transfer. 30s loop, repeated through the four main phases.  
P7: peak (155s-165s, beat: clear) — Drill: increase leading variety and expression. 10s.  
P8-P9: main (165s-207s, beat: clear/moderate) — Drill: same walk-step as P3-P6, but reduce travel and keep a small pulse in the body during the break. 42s.  
P10: main (207s-305s, beat: clear) — Drill: add subtle hip styling once the basic feels automatic. 30s loop, repeated through the main phases.  
P11: short_break (305s-315s, beat: clear) — Drill: reduce travel and reset your frame. 10s.  
P12: outro (315s-330s, beat: clear) — Drill: slow the pace and return to minimal movement. 15s.

### Transitions — what to do at boundaries

T1: 15s [intro → short_break, beat: clear → moderate] — as the energy lifts, transition from minimal movement to a steady walk-step.  
T2: 21s [short_break → main, beat: moderate → clear] — when the bass kicks in, find the pulse and start your walk-step.  
T3: 155s [main → peak, beat: clear → clear] — as the energy intensifies, commit to the new energy with stronger saídas and more expression.  
T4: 165s [peak → main, beat: clear → clear] — as the energy settles, breathe and return to a steady, sustaining walk-step.  
T5: 192s [main → break, beat: clear → moderate] — as the energy fades, keep a small pulse in the body and listen.  
T6: 207s [break → main, beat: moderate → clear] — walk-step on the first clear bass hit; don't chase the percussion.  
T7: 305s [main → short_break, beat: clear → clear] — as the energy fades and the percussion thins, contract your movement and slow the basic.  
T8: 315s [short_break → outro, beat: clear → clear] — prepare to close by returning to minimal movement and letting the dance end gently.

### Unified timeline (phases + transitions in one story)

```
P1: 0s-15s, intro [beat: clear] — The pulse is clear; focus on a close embrace and just feel the connection.
T1: 15s [intro → short_break, beat: clear → moderate] — as the energy lifts, transition from minimal movement to a steady walk-step.
P2: 15s-21s, short_break [beat: moderate] — The beat is moderate; stay compact and test the pulse with small steps.
T2: 21s [short_break → main, beat: moderate → clear] — when the bass kicks in, find the pulse and start your walk-step.
P3: 21s-64s, main ×4 [beat: clear] — Establishing the rhythm; it is safe to trust the steady pulse and walk the basic.
P4: 64s-107s, main ×4 [beat: clear] — Sustaining the energy; travel a little more while following the bass line.
P5: 107s-139s, main ×3 [beat: clear] — Sustaining the flow; keep your weight transfers smooth and steady.
P6: 139s-155s, main ×2 [beat: clear] — Building toward the peak; add more intention to your movement.
T3: 155s [main → peak, beat: clear → clear] — as the energy intensifies, commit to the new energy with stronger saídas and more expression.
P7: 155s-165s, peak [beat: clear] — The emotional high point; use the clear pulse to add more expression.
T4: 165s [peak → main, beat: clear → clear] — as the energy settles, breathe and return to a steady, sustaining walk-step.
P8: 165s-192s, main ×2 [beat: clear] — Returning to the basics; re-establish your connection with a steady walk.
T5: 192s [main → break, beat: clear → moderate] — as the energy fades, keep a small pulse in the body and listen.
P9: 192s-207s, break [beat: moderate] — The beat is moderate; reduce travel and reset your frame.
T6: 207s [break → main, beat: moderate → clear] — walk-step on the first clear bass hit; don't chase the percussion.
P10: 207s-305s, main ×9 [beat: clear] — Sustaining through the final stretch; keep the pulse automatic and walk evenly.
T7: 305s [main → short_break, beat: clear → clear] — as the energy fades and the percussion thins, contract your movement and slow the basic.
P11: 305s-315s, short_break [beat: clear] — The pulse is clear; use this moment to breathe and prepare to wind down.
T8: 315s [short_break → outro, beat: clear → clear] — prepare to close by returning to minimal movement and letting the dance end gently.
P12: 315s-330s, outro [beat: clear] — Closing the dance; contract your movement and let the song end gently.
```

## Bring your own audio

If you set `AUDIO_PATH` in the config cell and you are in `live` mode,
the cell below runs the full pipeline on your file.  Otherwise it is a
no-op.


In [8]:
if AUDIO_PATH and Path(AUDIO_PATH).exists():
    if MODE != "live":
        print("AUDIO_PATH set but MODE is not 'live' — set GEMMA_API_KEY and re-run cell 7.")
    else:
        print(f"Analyzing {AUDIO_PATH} live …")
        byo_audio = load_audio(AUDIO_PATH)
        byo_analysis = analyze(byo_audio, dance_style="kizomba")
        byo_outs = run_live_outputs(byo_analysis)
        display(Markdown(
            f"**DSP summary** — tempo {byo_analysis.tempo:.1f} BPM · "
            f"{len(byo_analysis.beats.times)} beats · "
            f"{len(getattr(byo_analysis, 'sections', []) or [])} sections"
        ))
        show_timeline(byo_analysis, title=Path(AUDIO_PATH).stem)
        for key, heading in [
            ("rhythm_anatomy",      "Rhythm anatomy"),
            ("listening_guide",     "Listening guide"),
            ("song_arc",            "Song arc"),
            ("kizomba_tutor",       "Kizomba tutor"),
            ("kizomba_drills",      "Drills"),
            ("kizomba_transitions", "Transitions"),
        ]:
            text = byo_outs.get(key, "")
            if text and not text.startswith("[ERROR"):
                display(Markdown(f"### {heading}"))
                display(Markdown(as_markdown(text)))
        if "unified_timeline" in byo_outs:
            display(Markdown("### Unified timeline"))
            display(Markdown("```\n" + byo_outs["unified_timeline"] + "\n```"))
else:
    print("No AUDIO_PATH set — skipping BYO cell.")
    print("To use your own audio: see the swap-in cell above.")

No AUDIO_PATH set — skipping BYO cell.
To use your own audio: see the swap-in cell above.


## What this demonstrates

- **Style-aware DSP.**  `analyze(audio, dance_style="kizomba")` runs a
  beat tracker tuned for the kizomba batida (low-pass + low-band onset
  envelope) and reports per-section *beat clarity tags* (subtle /
  moderate / clear) plus *downbeat confidence* so Gemma can judge how
  certain to sound.
- **Gemma 4 as the language layer.**  Six prompts cover genre context,
  listening orientation, song-arc narration, per-phase coaching, drills,
  and boundary transitions.  Each prompt is grounded in DSP features and
  carries hard rules (e.g. *"don't say 'the 1'"* for kizomba) baked
  directly into the prompt text.
- **Verifiers as guardrails.**  The drills and transitions prompts run
  through Python verifiers that drop or rewrite any Gemma output that
  invents sections, drops the P# format, or otherwise hallucinates.
- **Honest about uncertainty.**  When `downbeat_confidence` is low, the
  prompts explicitly tell Gemma to coach in *"feel the bass"* terms
  instead of count-based language.
- **Practical local-first stack.**  Default local target is Gemma 4 E2B /
  E4B via Ollama — small enough for a laptop, accurate enough for
  coaching.  Larger checkpoints help for the most stylized outputs but
  are never required.

> **Design principle:** *DSP grounds, Gemma writes, code verifies.*
> No layer is asked to do what another layer does better.


## Attribution & licences

### Audio — CC0 1.0 Universal (public domain)

The demo track is from the album **"Un Toque De To"**, released into
the public domain under
[CC0 1.0 Universal](https://creativecommons.org/publicdomain/zero/1.0/).
The work has no rights reserved: you may copy, modify, distribute, and
perform it, even for commercial purposes, without asking permission.

| # | Title | Source |
|---|---|---|
| 07 | *Besame Otra Vez*   | https://archive.org/details/un-toque-de-to |

### Model — Gemma 4

This notebook uses [Gemma 4](https://ai.google.dev/gemma) via the
OpenAI-compatible endpoint at
`https://generativelanguage.googleapis.com/v1beta/openai/`.  The default
slug in the config cell is `google/gemma-4-e4b-it` — small enough to
host on Kaggle's free T4.

> **Note on the cached outputs shown above:** the pre-generated Gemma
> text bundled with this notebook was produced with the larger
> `gemma-4-26b-a4b-it` checkpoint, which gives noticeably richer
> coaching language than E4B.  Running the notebook in *live* mode with
> `google/gemma-4-e4b-it` (the default) will produce shorter, simpler
> outputs — still grounded by the same DSP analysis and verifiers.
> Set `GEMMA_MODEL = "google/gemma-4-26b-a4b-it"` in the config cell to
> reproduce the cached quality live.

See `docs/project-vision.md` in the repo for model-selection guidance.

### Code — Rytmi

Rytmi's source code (this notebook, prompts, DSP pipeline, verifiers)
is in the public repository
[**mukatee/rytmi-pub**](https://github.com/mukatee/rytmi-pub).  See
`LICENSE` there for terms.

### Demo video

End-to-end walkthrough on real-world (copyrighted) kizomba and bachata
tracks: <https://youtu.be/ISkf6fZbG-Y>.  Educational use only; rights
remain with the original artists.

### Further reading

All links point to the public mirror at
[mukatee/rytmi-pub](https://github.com/mukatee/rytmi-pub).

- [`docs/kaggle_writeup.md`](https://github.com/mukatee/rytmi-pub/blob/master/docs/kaggle_writeup.md) —
  the Kaggle submission writeup, including the lessons-learned section
  on what worked and what didn't when prompting Gemma 4 for grounded
  music coaching (e.g. *"helper rationale text becomes echoed
  vocabulary"*, *"negative examples can backfire"*, *"code beats prompt
  prose for structural invariants"*).
- [`docs/how-it-works.md`](https://github.com/mukatee/rytmi-pub/blob/master/docs/how-it-works.md) —
  walkthrough of the DSP → prompt → Gemma architecture and the
  verifier layer.
- [`docs/project-vision.md`](https://github.com/mukatee/rytmi-pub/blob/master/docs/project-vision.md) —
  problem framing, model-selection guidance, and roadmap.
- [`docs/audio-sources.md`](https://github.com/mukatee/rytmi-pub/blob/master/docs/audio-sources.md) —
  how the CC0 demo track was chosen and licensing details for other
  audio paths.
- [`docs/experiments/`](https://github.com/mukatee/rytmi-pub/tree/master/docs/experiments) —
  dated phase notes capturing each iteration: prompt-tuning learnings,
  the kizomba batida tracker, the mel-filterbank gotcha, the drills
  verifier, the unified-timeline design, and the Gemma audio probe
  negative result.
